In [ ]:
# =============================================================================
# [Step 1] 환경 설정 & 라이브러리 설치
# =============================================================================
print("⏳ RAPIDS 라이브러리 설치 중... (약 2분 소요)")
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com > /dev/null
!pip install gdown > /dev/null
print("✅ 설치 완료.")

import gdown
import pandas as pd
import numpy as np
import math
import gc
import os
import shutil
import zipfile
from collections import defaultdict, Counter
from tqdm import tqdm
from cuml.ensemble import RandomForestClassifier as cuRF

# =============================================================================
# [Step 2] 데이터 파일 준비
# =============================================================================
zip_file_id = 'YOUR_DRIVE_FILE_ID'
target_zip = 'data.zip'
data_dir = 'data'

if not os.path.exists(target_zip):
    print(f"📥 데이터 다운로드 중...")
    url = f'https://drive.google.com/uc?id={zip_file_id}'
    gdown.download(url, target_zip, quiet=False)

if not os.path.exists(data_dir):
    os.makedirs(data_dir)

print("📦 압축 해제 중...")
with zipfile.ZipFile(target_zip, 'r') as zip_ref:
    zip_ref.extractall(data_dir)

# CSV 파일 위치 보정
required_files = ['train.csv', 'test.csv', 'sample_submission.csv']
for filename in required_files:
    target_path = os.path.join(data_dir, filename)
    if not os.path.exists(target_path):
        for root, dirs, files in os.walk(data_dir):
            if filename in files:
                shutil.move(os.path.join(root, filename), target_path)
                break
        if not os.path.exists(target_path) and os.path.exists(filename):
            shutil.move(filename, target_path)

print(f"📂 데이터 준비 완료: {os.listdir(data_dir)}")

# =============================================================================
# [Step 3] 자모 유틸리티
# =============================================================================
BASE_CODE = 0xAC00
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    result = []
    if not isinstance(text, str): return []
    # 입력 길이 제한 (안전장치)
    if len(text) > 100: text = text[:100]
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch)
        else:
            cho = code // 588
            jung = (code % 588) // 28
            jong = (code % 588) % 28
            result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    result = []
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            code = 0xAC00 + (CHOSUNG_LIST.index(cho)*588) + (JUNGSUNG_LIST.index(jung)*28) + JONGSUNG_LIST.index(jong)
            result.append(chr(code))
            i += 3
        else:
            result.append(cho)
            i += 1
    return "".join(result)

def align_jamo_list(src_list, tgt_list):
    # DP 테이블 크기 제한 (안전장치)
    MAX_LEN = 300
    if len(src_list) > MAX_LEN: src_list = src_list[:MAX_LEN]
    if len(tgt_list) > MAX_LEN: tgt_list = tgt_list[:MAX_LEN]

    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], '<DEL>')); i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1])); j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1])); i-=1; j-=1
    return alignment[::-1]

# =============================================================================
# [Step 4] 디스크 기반 데이터 생성 (메모리 보호)
# =============================================================================
def generate_dataset_disk_safe(csv_path, token_to_id, window_size=7):
    PAD = '<PAD>'
    pad_id = token_to_id.get(PAD, 0)

    print("👀 1차 스캔: 샘플 개수 파악 중...")
    total_samples = 0
    for chunk in pd.read_csv(csv_path, chunksize=5000, encoding='utf-8-sig'):
        for _, row in chunk.iterrows():
            in_jamos = to_jamo_list(row['input'])
            out_jamos = to_jamo_list(row['output'])
            # 대략적 계산으로 속도 향상
            total_samples += max(len(in_jamos), len(out_jamos)) + 2
        del chunk
        gc.collect()

    print(f"📝 할당할 최대 샘플 수: {total_samples:,}")

    X_shape = (total_samples, window_size)
    y_shape = (total_samples,)

    if os.path.exists('X_data.dat'): os.remove('X_data.dat')
    if os.path.exists('y_data.dat'): os.remove('y_data.dat')

    X_mm = np.memmap('X_data.dat', dtype='float32', mode='w+', shape=X_shape)
    y_mm = np.memmap('y_data.dat', dtype='float32', mode='w+', shape=y_shape)

    print("✍️ 2차 스캔: 디스크에 데이터 기록 중...")
    current_idx = 0
    for chunk in tqdm(pd.read_csv(csv_path, chunksize=2000, encoding='utf-8-sig'), desc="Disk Write"):
        for _, row in chunk.iterrows():
            in_jamos = to_jamo_list(row['input'])
            out_jamos = to_jamo_list(row['output'])
            alignment = align_jamo_list(in_jamos, out_jamos)

            in_ids = [token_to_id.get(t, 0) for t in in_jamos]
            padded_ids = [pad_id]*(window_size//2) + in_ids + [pad_id]*(window_size//2)

            curr = 0
            for src, tgt in alignment:
                if src is None: continue
                win = padded_ids[curr : curr + window_size]
                tgt_id = token_to_id.get(tgt, 0)
                if current_idx < total_samples:
                    X_mm[current_idx] = win
                    y_mm[current_idx] = tgt_id
                    current_idx += 1
                curr += 1
        if current_idx % 10000 == 0:
            X_mm.flush()

    print(f"✅ 데이터 생성 완료. 유효 샘플 수: {current_idx:,}")
    return X_mm[:current_idx], y_mm[:current_idx]

# =============================================================================
# [Step 5] 앙상블 학습 (Ensemble Learning) - 메모리 폭발 방지 핵심
# =============================================================================

# 1. 토큰 맵 구축
print("🏗️ 토큰 맵 구축 중...")
token_set = set(['<PAD>', '<DEL>'])
vocab_set = set()
for chunk in pd.read_csv('data/train.csv', chunksize=5000, encoding='utf-8-sig'):
    text_blob = "".join(chunk['input'].astype(str).tolist()) + "".join(chunk['output'].astype(str).tolist())
    if len(text_blob) > 50000: text_blob = text_blob[:50000]
    token_set.update(to_jamo_list(text_blob))
    for txt in chunk['output'].astype(str):
        for w in txt.split(): vocab_set.add(w)
    del chunk, text_blob
    gc.collect()

token_to_id = {t: i for i, t in enumerate(sorted(list(token_set)))}
id_to_token = {i: t for t, i in token_to_id.items()}
print(f"✅ 토큰 수: {len(token_to_id)}")

# 2. 데이터셋 로드 (Disk Memmap)
X_disk, y_disk = generate_dataset_disk_safe('data/train.csv', token_to_id, window_size=7)
total_len = len(X_disk)
print(f"📊 총 학습 데이터: {total_len:,}개")

# 3. 분할 학습 (Divide and Conquer)
# 170만개를 5개로 나누면 약 34만개씩. 이 정도는 T4 GPU에서 절대 안 터집니다.
N_SPLITS = 5
models = []

print(f"🌲 안전 모드: 데이터를 {N_SPLITS}개로 나누어 순차 학습합니다.")

chunk_size = math.ceil(total_len / N_SPLITS)
for i in range(N_SPLITS):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, total_len)

    if start >= end: break

    print(f"   ▶ 모델 {i+1}/{N_SPLITS} 학습 중... (데이터 범위: {start:,} ~ {end:,})")

    # 해당 구간만 슬라이싱 (메모리 복사 없음)
    X_chunk = X_disk[start:end]
    y_chunk = y_disk[start:end]

    # 작은 모델 생성 (나무 60개 * 5모델 = 총 300개 효과)
    model = cuRF(
        n_estimators=60,
        max_depth=22,
        max_features=1.0,
        n_bins=32,
        random_state=42 + i, # 서로 다른 랜덤 시드
        verbose=False
    )

    model.fit(X_chunk, y_chunk)
    models.append(model)

    # 사용한 청크 메모리 해제 시도
    del X_chunk, y_chunk, model
    gc.collect()

print("✅ 모든 모델 학습 완료!")

# 디스크 파일 정리
del X_disk, y_disk
gc.collect()
if os.path.exists('X_data.dat'): os.remove('X_data.dat')
if os.path.exists('y_data.dat'): os.remove('y_data.dat')


# =============================================================================
# [Step 6] 앙상블 추론 (수정됨: 확률표 크기 자동 보정)
# =============================================================================
import numpy as np
import math
from tqdm import tqdm
import gc

def decode_ensemble(text, models, token_to_id, id_to_token, lm_prob, vocab_set,
                    window_size=7, beam_size=10, lm_weight=2.5, threshold=3.0):
    if not isinstance(text, str): return ""
    in_jamos = to_jamo_list(text)
    pad_id = token_to_id.get('<PAD>', 0)

    in_ids = [token_to_id.get(t, 0) for t in in_jamos]
    padded_ids = [pad_id]*(window_size//2) + in_ids + [pad_id]*(window_size//2)

    X_enc = []
    for i in range(len(in_jamos)):
        X_enc.append(padded_ids[i : i + window_size])

    if not X_enc: return text

    # GPU 추론을 위한 입력 변환
    X_enc_gpu = np.array(X_enc, dtype=np.float32)

    # -------------------------------------------------------------------------
    # [핵심 수정] 모델마다 아는 글자 수가 달라도 합칠 수 있게 보정
    # -------------------------------------------------------------------------
    n_total_classes = len(token_to_id) # 전체 자모 개수 (예: 53)
    avg_probs = np.zeros((len(X_enc_gpu), n_total_classes), dtype=np.float32)

    for model in models:
        # 1. 모델 예측 (GPU -> CPU)
        probs = model.predict_proba(X_enc_gpu)
        if hasattr(probs, 'get'): probs = probs.get()

        # 2. 이 모델이 학습한 글자들의 ID 목록 가져오기
        m_classes = model.classes_
        if hasattr(m_classes, 'get'): m_classes = m_classes.get()
        m_classes = m_classes.astype(int)

        # 3. 전역 확률표의 '제자리'에 값 채워넣기
        # (모델이 배우지 못한 글자는 확률 0으로 유지됨)
        avg_probs[:, m_classes] += probs

    avg_probs /= len(models) # 평균 내기
    # -------------------------------------------------------------------------

    # Beam Search 로직 (기존과 동일)
    beam = [([], None, 0.0)]
    for i in range(len(in_jamos)):
        curr_probs = avg_probs[i]
        top_k_indices = np.argsort(curr_probs)[-beam_size:]
        new_beam = []
        for (decoded, last_char, score) in beam:
            for idx in top_k_indices:
                char_id = int(idx)
                char_prob = float(curr_probs[idx])
                if char_prob <= 0: char_prob = 1e-9
                pred_char = id_to_token.get(char_id, '')
                ml_score = math.log(char_prob)
                lm_p = lm_prob.get((last_char, pred_char), 1e-7)
                lm_score = math.log(lm_p)
                total_score = score + ml_score + (lm_weight * lm_score)
                if pred_char == '<DEL>': new_beam.append((decoded, last_char, total_score))
                else: new_beam.append((decoded + [pred_char], pred_char, total_score))
        new_beam.sort(key=lambda x: x[2], reverse=True)
        beam = new_beam[:beam_size]

    candidates = []
    for (decoded, _, sc) in beam:
        word = from_jamo_list(decoded)
        candidates.append((word, sc))

    if not candidates: return text
    best_word, best_score = candidates[0]
    if best_word in vocab_set: return best_word
    for i in range(1, len(candidates)):
        cand_word, cand_score = candidates[i]
        if best_score - cand_score > threshold: break
        if cand_word in vocab_set: return cand_word
    return best_word

print("🚀 최종 추론 다시 시작...")
sample_submission = pd.read_csv('data/sample_submission.csv', encoding='utf-8-sig')
outputs = []
test_chunk_iter = pd.read_csv('data/test.csv', chunksize=1000, encoding='utf-8-sig')

for chunk in tqdm(test_chunk_iter, desc="Inferencing"):
    for idx, row in chunk.iterrows():
        obf_text = row['input']
        tokens = obf_text.split()
        new_tokens = []
        for t in tokens:
            # 수정된 함수 호출
            restored = decode_ensemble(t, models, token_to_id, id_to_token, lm_prob, vocab_set)
            new_tokens.append(restored)
        outputs.append(" ".join(new_tokens))
    gc.collect()

sample_submission['output'] = outputs
sample_submission.to_csv('submission_final_gpu.csv', index=False)
print("🎉 진짜 완료! 'submission_final_gpu.csv' 저장됨.")